# XCiT Model Checkpoint Loading and Weight Extraction

This notebook demonstrates how to:
1. Load a PyTorch Lightning checkpoint (`.ckpt`) for inference.
2. Extract only the model weights from a checkpoint and save them as a standard PyTorch `.pt` file.
3. Load the weights from the `.pt` file into a model instance.

In [ ]:
import torch
from torchsig_models.models.iq_models.xcit.xcit1d import XCiTClassifier
import pytorch_lightning as pl

# Path to your checkpoint file
ckpt_path = "path/to/your/checkpoint.ckpt" # Replace with actual path

print(f"Loading checkpoint from: {ckpt_path}")

## 1. Load Checkpoint for Inference

PyTorch Lightning provides a convenient `load_from_checkpoint` method that reconstructs the model with the saved hyperparameters and weights.

In [ ]:
try:
    # Load the full LightningModule
    model = XCiTClassifier.load_from_checkpoint(ckpt_path)
    model.eval()
    model.freeze()
    print("Model successfully loaded from checkpoint!")
except Exception as e:
    print(f"Error loading checkpoint: {e}")
    print("Note: Ensure ckpt_path is correct and the XCiTClassifier class is available.")

## 2. Extract and Save Only Model Weights

Checkpoint files contain more than just weights (optimizer state, epoch, hyperparameters, etc.). To save only the model weights to a `.pt` file, we extract the `state_dict`.

In [ ]:
weights_path = "xcit_weights.pt"

try:
    # Load the checkpoint as a dictionary
    checkpoint = torch.load(ckpt_path, map_location=torch.device('cpu'))
    
    # The model weights are stored under 'state_dict'
    state_dict = checkpoint['state_dict']
    
    # Save only the state_dict to a .pt file
    torch.save(state_dict, weights_path)
    print(f"Model weights extracted and saved to: {weights_path}")
except Exception as e:
    print(f"Error extracting weights: {e}")

## 3. Load Weights from .pt File

When loading from a `.pt` file, you must first instantiate the model with the correct hyperparameters, then load the state dictionary.

In [ ]:
try:
    # You must know the hyperparameters used during training
    # These can be found in the .ckpt file if you don't have them:
    # hyperparameters = torch.load(ckpt_path)['hyper_parameters']
    
    # Example hyperparameters (adjust based on your training config)
    input_channels = 2
    num_classes = 57 # Adjust to your actual number of classes
    
    # 1. Instantiate the model
    model_from_pt = XCiTClassifier(input_channels=input_channels, num_classes=num_classes)
    
    # 2. Load the state dict
    state_dict = torch.load(weights_path, map_location=torch.device('cpu'))
    
    # 3. Load weights into the model
    model_from_pt.load_state_dict(state_dict)
    model_from_pt.eval()
    
    print("Model weights successfully loaded from .pt file!")
except Exception as e:
    print(f"Error loading weights from .pt file: {e}")

### Summary of Differences

| Method | File Format | Contains | Ease of Use |
| :--- | :--- | :--- | :--- |
| `load_from_checkpoint` | `.ckpt` | Weights, Hyperparams, Optimizer state | Very High (Automated) |
| `torch.save(state_dict)` | `.pt` | Weights only | Medium (Requires manual model setup) |